In [ ]:
# ============================================================================
# Kuntal Ghosh
# Early Feb 2026
# Results not yet ready/published!

# This is to use VAMPNet (Noe, 2018) for better understanding the behavior/
# curvature of a "hexamer-of-hexamer" capsid patch

# ============================================================================
import numpy as np
import MDAnalysis as mda
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from deeptime.decomposition.deep import VAMPNet
from deeptime.util.data import TrajectoryDataset
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from scipy.ndimage import gaussian_filter
from matplotlib.colors import LinearSegmentedColormap
from itertools import combinations

In [ ]:
# ============================================================================
# Configuration
# ============================================================================
REFERENCE_PDB = "../CPSF6_fresh/ca_cpsf6_nup153.pdb"
traj_paths = [
    ("../CPSF6_fresh/ca_cpsf6_nup153.pdb",      "../CPSF6_fresh/replica_0/traj.xtc"),
    ("../CPSF6_fresh/ca_cpsf6_nup153.pdb",      "../CPSF6_fresh/replica_1/traj.xtc"),
    ("../CPSF6_fresh/ca_cpsf6_nup153.pdb",      "../CPSF6_fresh/replica_2/traj.xtc"),
    ("../../ca_nup153/main_run/first_frame.pdb", "../../ca_nup153/main_run/traj.xtc"),
    ("../../ca_nup153/main_run/first_frame.pdb", "../../ca_nup153/replica_1/traj.xtc"),
    ("../../ca_nup153/main_run/first_frame.pdb", "../../ca_nup153/replica_2/traj.xtc"),
]

LAG          = 20
BATCH_SIZE   = 512
N_EPOCHS     = 500
LR           = 3e-4
L2           = 1e-3
N_STATES     = 5   # This seems to be the best in terms of training a good model - discard states not visited
N_ENSEMBLE   = 3
SEED         = 36
FRAME_TIME   = 1.0   # 1 ns per frame printed
kT_310       = 0.008314/4.184 * 310
STATE_NAMES  = [chr(65 + k) for k in range(N_STATES)]   # ["A","B","C","D","E"]
state_colors = ["#e41a1c", "#377eb8", "#4daf4a", "#ff7f00", "#984ea3"][:N_STATES]
replica_labels = ["CPSF6 rep0", "CPSF6 rep1", "CPSF6 rep2",
                  "Bare  rep0", "Bare  rep1", "Bare  rep2"]

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ============================================================================
# Hexamer definitions
# ============================================================================
hexamer1 = ['A', 'B', 'C', 'D', 'E', 'F']
hexamer2 = ['G', 'H', 'I', 'J', 'K', 'L']
hexamer3 = ['M', 'N', 'O', 'P', 'Q', 'R']
hexamer4 = ['S', 'T', 'U', 'V', 'W', 'X']
hexamer5 = ['Y', 'Z', 'a', 'b', 'c', 'd']
hexamer6 = ['e', 'f', 'g', 'h', 'i', 'j']
hexamer_pairs = [
    (hexamer1, hexamer2), (hexamer1, hexamer3),
    (hexamer2, hexamer4), (hexamer3, hexamer4),
    (hexamer3, hexamer5), (hexamer4, hexamer6),
    (hexamer5, hexamer6),
]
all_hexamers       = [hexamer1, hexamer2, hexamer3, hexamer4, hexamer5, hexamer6]
ALL_HEXAMER_CHAINS = [ch for hx in all_hexamers for ch in hx]
pair_labels_base   = ["H1-H2", "H1-H3", "H2-H4",
                      "H3-H4", "H3-H5", "H4-H6", "H5-H6"]
n_pairs            = len(pair_labels_base)

In [ ]:
# ============================================================================
# Geometry utilities
# ============================================================================
# Fitting planes to each hexamer and then computing angles between normal vectors
# Seems to be the best feature for now, let's see
def fit_plane_normal(points):
    c = points - points.mean(axis=0)
    _, _, vh = np.linalg.svd(c)
    n = vh[-1]
    return n / np.linalg.norm(n)

def angle_between_vectors(v1, v2):
    v1 = v1 / np.linalg.norm(v1)
    v2 = v2 / np.linalg.norm(v2)
    return float(np.arccos(np.clip(np.dot(v1, v2), -1.0, 1.0)))

# Kabsch rotation here is equivalent to using RMSD trajectory align tool on VMD
# GROMACS input trajectory was extracted using -pbc nojump
def kabsch_rotation(mobile_c, ref_c):
    H        = mobile_c.T @ ref_c
    U, _, Vt = np.linalg.svd(H)
    d        = np.linalg.det(Vt.T @ U.T)
    D        = np.diag([1.0, 1.0, d])
    return Vt.T @ D @ U.T

# ============================================================================
# Reference universe
# ============================================================================
_chain_sel = " or ".join(f"segid {ch}" for ch in ALL_HEXAMER_CHAINS)
ALIGN_SEL  = f"name CA and ({_chain_sel})"

print(f"Loading reference: {REFERENCE_PDB}")
ref_u        = mda.Universe(REFERENCE_PDB)
ref_ag       = ref_u.select_atoms(ALIGN_SEL)
ref_com      = ref_ag.positions.mean(axis=0).copy()
ref_centered = (ref_ag.positions - ref_com).copy()
print(f"  Reference CA atoms: {len(ref_ag)}")

In [ ]:
# ============================================================================
# Feature extraction
# For VAMP or VAMPNets (for TICA as well actually) it's difficult to use raw
# angles as inputs, better use sin/cosines of angles
# So 7 pairs of angles between hexamers -> 14 input features
# ============================================================================
def compute_angles_cossin(traj_path: tuple) -> np.ndarray:
    topology, trajectory = traj_path
    system_label = "CA-NUP153-CPSF6" if "CPSF6" in topology else "CA-NUP153"
    print(f"\n  [{system_label}] {trajectory}")
    u      = mda.Universe(topology, trajectory)
    segids = {seg.segid for seg in u.segments}
    missing = [ch for ch in ALL_HEXAMER_CHAINS if ch not in segids]
    if missing:
        raise ValueError(
            f"Chains {missing} not found in {trajectory}.\n"
            f"  Available segids: {sorted(segids)}"
        )
    n_frames   = len(u.trajectory)
    n_hexamers = len(all_hexamers)
    mobile_ag  = u.select_atoms(ALIGN_SEL)
    if len(mobile_ag) != len(ref_ag):
        raise ValueError(
            f"Atom count mismatch: mobile={len(mobile_ag)} "
            f"reference={len(ref_ag)}"
        )
    mobile_index_map = {global_idx: local_idx
                        for local_idx, global_idx in enumerate(mobile_ag.indices)}
    chain_indices_local = [
        [np.array([mobile_index_map[i]
                   for i in u.select_atoms(f"segid {ch} and name CA").indices])
         for ch in hx]
        for hx in all_hexamers
    ]
    chain_masses = [
        [u.select_atoms(f"segid {ch} and name CA").masses for ch in hx]
        for hx in all_hexamers
    ]
    hexamer_chain_coms = np.zeros((n_frames, n_hexamers, 6, 3))
    for i, ts in enumerate(u.trajectory):
        mob_pos = mobile_ag.positions.copy()
        mob_com = mob_pos.mean(axis=0)
        mob_c   = mob_pos - mob_com
        R       = kabsch_rotation(mob_c, ref_centered)
        aligned_mobile = mob_c @ R.T
        for h, (hx_idx, hx_masses) in enumerate(
                zip(chain_indices_local, chain_masses)):
            for c, (idx, masses) in enumerate(zip(hx_idx, hx_masses)):
                pos_c = aligned_mobile[idx]
                hexamer_chain_coms[i, h, c] = (
                    (pos_c * masses[:, None]).sum(axis=0) / masses.sum()
                )
    print(f"    Aligned {n_frames} frames OK")
    normals = np.zeros((n_frames, n_hexamers, 3))
    for i in range(n_frames):
        for h in range(n_hexamers):
            normals[i, h] = fit_plane_normal(hexamer_chain_coms[i, h])
    angles_cossin = []
    for hx1, hx2 in hexamer_pairs:
        idx1 = all_hexamers.index(hx1)
        idx2 = all_hexamers.index(hx2)
        ang  = np.array([
            angle_between_vectors(normals[i, idx1], normals[i, idx2])
            for i in range(n_frames)
        ])
        angles_cossin.append(np.column_stack([np.sin(ang), np.cos(ang)]))
    return np.hstack(angles_cossin)   # (n_frames, 14)

print("\n" + "="*60)
print("Feature extraction")
print("="*60)
all_features_list = [compute_angles_cossin(p) for p in traj_paths]
all_features_f32  = [f.astype(np.float32) for f in all_features_list]
INPUT_DIM         = all_features_list[0].shape[1]
print(f"\nInput dimensionality: {INPUT_DIM}")
print("\nFeature shapes:")
for i, ((_, traj), feat) in enumerate(zip(traj_paths, all_features_list)):
    label = "CPSF6" if i < 3 else "bare "
    print(f"  [{label}] {feat.shape}  {traj}")

In [ ]:
# ============================================================================
# Dataset
# ============================================================================
dataset = TrajectoryDataset.from_trajectories(lagtime=LAG, data=all_features_f32)
n_total = len(dataset)
n_train = int(0.8 * n_total)
n_val   = n_total - n_train
train_data, val_data = torch.utils.data.random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE,
                          shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE,
                          shuffle=False, drop_last=False)
val_x0 = torch.cat([b[0] for b in val_loader], dim=0).to(DEVICE)
val_x1 = torch.cat([b[1] for b in val_loader], dim=0).to(DEVICE)
print(f"Dataset: {n_total} pairs | Train: {n_train} | Val: {n_val}")

In [ ]:
# ============================================================================
# Architecture
# I tested with some other layer configurations, this seems to be the best
# ============================================================================
class VAMPEncoder(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, output_dim),
            nn.Softmax(dim=-1),
        )

    def forward(self, x):
        return self.network(x)

# ============================================================================
# Ensemble training with LR scheduler
# ============================================================================
print(f"\n{'='*60}")
print(f"VAMPNet ensemble training | device={DEVICE} | "
      f"states={N_STATES} | ensemble={N_ENSEMBLE}")
print(f"{'='*60}")

# I'm taking three sets of VAMPNets, and choosing the one with the best score
# Training the NN was trickier than simple VAMP training though

trained_lobes    = []
all_train_curves = []
all_val_curves   = []
best_val_scores  = []

for member in range(N_ENSEMBLE):
    print(f"\n  [Ensemble member {member+1}/{N_ENSEMBLE}]")
    print(f"  {'Epoch':>6}  {'Train VAMP-2':>12}  {'Val VAMP-2':>10}  {'LR':>10}")
    print(f"  {'-'*44}")

    lobe    = VAMPEncoder(INPUT_DIM, N_STATES).to(DEVICE)
    vampnet = VAMPNet(lobe=lobe, optimizer="Adam",
                      learning_rate=LR, device=DEVICE, score_method="VAMP2")

    for pg in vampnet.optimizer.param_groups:
        pg["weight_decay"] = L2

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        vampnet.optimizer,
        mode     = "max",
        patience = 20,
        factor   = 0.5,
        min_lr   = 1e-6,
    )

    best_val   = -np.inf
    best_state = None
    train_eps  = []
    val_eps    = []

    for epoch in range(1, N_EPOCHS + 1):
        vampnet.fit(train_loader)
        tr = float(np.mean(vampnet.train_scores[-len(train_loader):]))
        vl = float(vampnet.validate((val_x0, val_x1)))
        train_eps.append(tr)
        val_eps.append(vl)

        scheduler.step(vl)
        current_lr = vampnet.optimizer.param_groups[0]["lr"]

        if vl > best_val:
            best_val   = vl
            best_state = {k: v.cpu().clone()
                          for k, v in lobe.state_dict().items()}
        if epoch % 50 == 0 or epoch == 1:
            tag = " ← best" if vl == best_val else ""
            print(f"  {epoch:>6}  {tr:>12.4f}  {vl:>10.4f}  "
                  f"{current_lr:>10.2e}{tag}")

    trained_lobes.append(best_state)
    all_train_curves.append(train_eps)
    all_val_curves.append(val_eps)
    best_val_scores.append(best_val)
    print(f"  Best val VAMP-2: {best_val:.4f}")

print(f"\nEnsemble training complete.")
print(f"Val VAMP-2 per member: {[f'{s:.4f}' for s in best_val_scores]}")
print(f"Mean ± std: {np.mean(best_val_scores):.4f} ± {np.std(best_val_scores):.4f}")
for i, sd in enumerate(trained_lobes):
    torch.save(sd, f"vampnet_lobe_member{i+1}.pt")
print("Weights saved.")

In [ ]:
# ============================================================================
# Training / validation curve
# ============================================================================
epochs    = np.arange(1, N_EPOCHS + 1)
train_arr = np.array(all_train_curves)
val_arr   = np.array(all_val_curves)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for arr, label, color in [
    (train_arr, "Train", "steelblue"),
    (val_arr,   "Val",   "darkorange"),
]:
    mean = arr.mean(axis=0);  std = arr.std(axis=0)
    for i in range(N_ENSEMBLE):
        axes[0].plot(epochs, -arr[i], color=color, lw=0.6, alpha=0.3)
        axes[1].plot(epochs,  arr[i], color=color, lw=0.6, alpha=0.3)
    axes[0].plot(epochs, -mean, color=color, lw=2.0, label=f"{label} mean")
    axes[0].fill_between(epochs, -(mean+std), -(mean-std), color=color, alpha=0.15)
    axes[1].plot(epochs,  mean, color=color, lw=2.0, label=f"{label} mean")
    axes[1].fill_between(epochs,  mean-std,   mean+std,   color=color, alpha=0.15)

axes[0].set_xlabel("Epoch", fontsize=13)
axes[0].set_ylabel("Loss (−VAMP2)", fontsize=13)
axes[0].set_title("Training Loss", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=11);  axes[0].grid(linestyle="--", alpha=0.4)

axes[1].set_xlabel("Epoch", fontsize=13)
axes[1].set_ylabel("VAMP-2 score", fontsize=13)
axes[1].set_title("VAMP-2 Score", fontsize=13, fontweight="bold")
axes[1].grid(linestyle="--", alpha=0.4)

best_idx = int(np.argmax(best_val_scores))
for i, score in enumerate(best_val_scores):
    axes[1].axhline(score, color="gray", lw=0.5, linestyle=":")
axes[1].axhline(best_val_scores[best_idx], color="green", lw=1.5,
                linestyle="--",
                label=f"Best member {best_idx+1} ({best_val_scores[best_idx]:.4f})")
axes[1].legend(fontsize=10)

plt.suptitle(f"VAMPNet Training | lag={LAG} | states={N_STATES} | "
             f"L2={L2} | ReduceLROnPlateau",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("vampnet_training_curve.png", dpi=150)
plt.show()
print("Saved: vampnet_training_curve.png")

In [ ]:
# ============================================================================
# Projection — best ensemble member
# ============================================================================
best_idx  = int(np.argmax(best_val_scores))
print(f"\nBest member: {best_idx+1} (val VAMP-2 = {best_val_scores[best_idx]:.4f})")

best_lobe = VAMPEncoder(INPUT_DIM, N_STATES).to(DEVICE)
best_lobe.load_state_dict(trained_lobes[best_idx])
best_lobe.eval()

vampnet_best = VAMPNet(lobe=best_lobe, device=DEVICE, score_method="VAMP2")
model        = vampnet_best.fetch_model()

all_soft   = [model.transform(feat) for feat in all_features_f32]
all_dtrajs = [np.argmax(p, axis=1) for p in all_soft]

all_state_ids = np.concatenate(all_dtrajs,     axis=0)
ids_cpsf6     = np.concatenate(all_dtrajs[:3], axis=0)
ids_bare      = np.concatenate(all_dtrajs[3:], axis=0)
soft_all      = np.concatenate(all_soft,        axis=0)
soft_cpsf6    = np.concatenate(all_soft[:3],    axis=0)
soft_bare     = np.concatenate(all_soft[3:],    axis=0)

print(f"\nRaw state populations (before filtering):")
for k in range(N_STATES):
    n = (all_state_ids == k).sum()
    print(f"  Raw state {STATE_NAMES[k]}: {n} ({100*n/len(all_state_ids):.1f}%)")

# ============================================================================
# Filter unpopulated states and relabel alphabetically
# I chose 5 states to begin with for better training -> it is okay to 
# discard the empty/unvisited states (one state here)
# ============================================================================
min_frames = 10

# Identify active raw state indices
active_raw = [k for k in range(N_STATES)
              if (ids_cpsf6 == k).sum() + (ids_bare == k).sum() >= min_frames]
dropped    = [k for k in range(N_STATES) if k not in active_raw]

print(f"\nActive raw states : {[STATE_NAMES[k] for k in active_raw]}")
print(f"Dropped raw states: {[STATE_NAMES[k] for k in dropped]}")

# Relabel: active_raw[0] -> "A", active_raw[1] -> "B", etc.
N_ACTIVE      = len(active_raw)
ACTIVE_NAMES  = [chr(65 + i) for i in range(N_ACTIVE)]   # ["A","B","C","D",...]
ACTIVE_COLORS = [state_colors[i] for i in range(N_ACTIVE)]

# Remap raw index -> new index (dropped states get -1)
raw_to_new = {raw: new for new, raw in enumerate(active_raw)}

print(f"\nRelabeling:")
for raw, new in raw_to_new.items():
    print(f"  Raw state {STATE_NAMES[raw]} -> State {ACTIVE_NAMES[new]}")

# Remap dtrajs — drop frames assigned to inactive states
def remap_dtraj(dtraj):
    new = np.full_like(dtraj, -1)
    for raw, new_idx in raw_to_new.items():
        new[dtraj == raw] = new_idx
    return new

all_dtrajs_active = [remap_dtraj(d) for d in all_dtrajs]

# Reindex soft probability arrays — keep only active state columns
soft_all_active    = soft_all[:,   active_raw]
soft_cpsf6_active  = soft_cpsf6[:, active_raw]
soft_bare_active   = soft_bare[:,  active_raw]

# Hard assignments using new indices — drop frames with label -1
ids_all_active   = np.concatenate(all_dtrajs_active,     axis=0)
ids_cpsf6_active = np.concatenate(all_dtrajs_active[:3], axis=0)
ids_bare_active  = np.concatenate(all_dtrajs_active[3:], axis=0)

# Valid frame masks (exclude dropped state frames)
mask_all_valid   = ids_all_active   >= 0
mask_cpsf6_valid = ids_cpsf6_active >= 0
mask_bare_valid  = ids_bare_active  >= 0

ids_all_active   = ids_all_active[mask_all_valid]
ids_cpsf6_active = ids_cpsf6_active[mask_cpsf6_valid]
ids_bare_active  = ids_bare_active[mask_bare_valid]

print(f"\nFrames after filtering:")
print(f"  All    : {mask_all_valid.sum()}")
print(f"  CPSF6  : {mask_cpsf6_valid.sum()}")
print(f"  NUP153 : {mask_bare_valid.sum()}")

print(f"\nRelabeled state populations:")
for i in range(N_ACTIVE):
    n = (ids_all_active == i).sum()
    print(f"  State {ACTIVE_NAMES[i]}: {n} ({100*n/len(ids_all_active):.1f}%)")

# ============================================================================
# Normalized frequency heatmaps — pairwise state probabilities
# Active states only, relabeled A-D
# ============================================================================
from itertools import combinations
from scipy.ndimage import gaussian_filter
from matplotlib.colors import LinearSegmentedColormap

hot_white = LinearSegmentedColormap.from_list(
    "hot_white",
    ["white", "#FFDD00", "#FF6600", "#CC0000", "#660000"],
    N=512,
)

pairs     = list(combinations(range(N_ACTIVE), 2))
n_pairs_p = len(pairs)

cond_data = [
    (soft_all_active,   "All Replicas"),
    (soft_cpsf6_active, "CPSF6-bound"),
    (soft_bare_active,  "NUP153 only"),
]

N_BINS = 100
SIGMA  = 1.5

fig, axes = plt.subplots(n_pairs_p, 3,
                         figsize=(19, 6 * n_pairs_p),
                         squeeze=False)

for row, (i, j) in enumerate(pairs):
    si, sj = ACTIVE_NAMES[i], ACTIVE_NAMES[j]

    pad    = 0.02
    xi_min = max(soft_all_active[:, i].min() - pad, 0.0)
    xi_max = min(soft_all_active[:, i].max() + pad, 1.0)
    xj_min = max(soft_all_active[:, j].min() - pad, 0.0)
    xj_max = min(soft_all_active[:, j].max() + pad, 1.0)
    xlim   = (xi_min, xi_max)
    ylim   = (xj_min, xj_max)

    for col, (data, title) in enumerate(cond_data):
        ax = axes[row, col]

        H, xedges, yedges = np.histogram2d(
            data[:, i], data[:, j],
            bins=N_BINS,
            range=[xlim, ylim],
        )
        H = gaussian_filter(H, sigma=SIGMA)
        H = H / H.max()
        H = H ** 0.5

        im = ax.imshow(
            H.T,
            origin="lower",
            extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
            aspect="auto",
            cmap=hot_white,
            vmin=0, vmax=1,
        )
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Normalized frequency", fontsize=10)

        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xlabel(f"P(State {si})", fontsize=12)
        ax.set_ylabel(f"P(State {sj})", fontsize=12)
        if row == 0:
            ax.set_title(title, fontsize=14, fontweight="bold")
        ax.text(0.02, 0.97, f"{si} vs {sj}",
                transform=ax.transAxes, fontsize=11,
                va="top", ha="left",
                bbox=dict(facecolor="white", edgecolor="gray",
                          alpha=0.7, boxstyle="round,pad=0.2"))

plt.suptitle("VAMPNet Pairwise State Probability Landscapes — Normalized Frequency",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("vampnet_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: vampnet_heatmaps.png")

In [ ]:
# ============================================================================
# State Population Distributions — relabeled active states
# ============================================================================
from scipy.stats import chi2_contingency

pop_cpsf6 = np.array([(ids_cpsf6_active == i).mean() for i in range(N_ACTIVE)])
pop_bare  = np.array([(ids_bare_active  == i).mean() for i in range(N_ACTIVE)])

counts_cpsf6 = np.array([(ids_cpsf6_active == i).sum() for i in range(N_ACTIVE)])
counts_bare  = np.array([(ids_bare_active  == i).sum() for i in range(N_ACTIVE)])

print(f"\n  State counts:")
for i in range(N_ACTIVE):
    print(f"    State {ACTIVE_NAMES[i]}: CPSF6={counts_cpsf6[i]}  NUP153={counts_bare[i]}")

counts_cpsf6_safe = counts_cpsf6 + 1
counts_bare_safe  = counts_bare  + 1
chi2, pval, dof, _ = chi2_contingency(
    np.vstack([counts_cpsf6_safe, counts_bare_safe])
)
print(f"\nChi-squared test (with pseudocount): chi2={chi2:.3f}, dof={dof}, p={pval:.2e}")
if pval < 0.05:
    print(f"→ Populations significantly different (p < 0.05)")
else:
    print(f"→ No significant difference (p >= 0.05)")

fig, ax = plt.subplots(figsize=(8, 6))
x = np.arange(N_ACTIVE)
w = 0.35

ax.bar(x - w/2, pop_cpsf6, w, label="CPSF6-bound",
       color="steelblue", edgecolor="k", linewidth=0.7)
ax.bar(x + w/2, pop_bare,  w, label="NUP153 only",
       color="coral",      edgecolor="k", linewidth=0.7)

for i in range(N_ACTIVE):
    ax.text(i - w/2, pop_cpsf6[i] + 0.005, f"{pop_cpsf6[i]:.2f}",
            ha="center", va="bottom", fontsize=11,
            color="steelblue", fontweight="bold")
    ax.text(i + w/2, pop_bare[i]  + 0.005, f"{pop_bare[i]:.2f}",
            ha="center", va="bottom", fontsize=11,
            color="coral", fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(ACTIVE_NAMES, fontsize=13)
ax.set_xlabel("State", fontsize=13)
ax.set_ylabel("Population fraction", fontsize=13)
ax.set_title("State Populations", fontsize=13)
ax.legend(fontsize=12, loc="upper left")
ax.set_ylim(0, 1.0)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("vampnet_state_populations.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: vampnet_state_populations.png")

In [ ]:
# ============================================================================
# Tilt distributions per state — 2 panels
# Left: CPSF6-bound | Right: NUP153 only
# KDE curves per panel (one per active state), colored by state
# Tilt averaged over all 7 pairs per frame
# Rep 0 and rep 1 only (need to fix this! NUP153 rep 2 is a little weird)
# VAMPNet states — active states only, relabeled A-D
# ============================================================================
from scipy.stats import gaussian_kde
from matplotlib.patches import Patch

N_PAIRS = 7

def recover_tilt_angles(features):
    angles = np.zeros((features.shape[0], N_PAIRS))
    for k in range(N_PAIRS):
        angles[:, k] = np.degrees(np.arccos(
            np.clip(features[:, 2*k + 1], -1.0, 1.0)
        ))
    return angles

# Recompute tilt angles from raw features
all_angles        = [recover_tilt_angles(f) for f in all_features_list]
angles_cpsf6_list = all_angles[:3]
angles_bare_list  = all_angles[3:]

# Average tilt over all 7 pairs per frame — rep 0 and rep 1 only
tilt_cpsf6_all    = [ang.mean(axis=1) for ang in angles_cpsf6_list[:2]]
tilt_bare_all     = [ang.mean(axis=1) for ang in angles_bare_list[:2]]
tilt_cpsf6_pooled = np.concatenate(tilt_cpsf6_all, axis=0)
tilt_bare_pooled  = np.concatenate(tilt_bare_all,  axis=0)

# Use VAMPNet active state dtrajs — rep 0 and rep 1 only
dtrajs_cpsf6_active = all_dtrajs_active[:3]
dtrajs_bare_active  = all_dtrajs_active[3:]

# Fixed x range
x_min   = 20
x_max   = 140
x_range = np.linspace(x_min, x_max, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

panel_configs = [
    (tilt_cpsf6_pooled, dtrajs_cpsf6_active[:2], "CPSF6-bound"),
    (tilt_bare_pooled,  dtrajs_bare_active[:2],  "NUP153 only"),
]

for ax, (tilt_pooled, dtrajs_list, title) in zip(axes, panel_configs):
    for i in range(N_ACTIVE):
        # Mask for active state i — exclude dropped frames (label -1)
        dtraj_concat = np.concatenate([d for d in dtrajs_list], axis=0)
        mask   = dtraj_concat == i
        tilt_k = tilt_pooled[mask]

        if len(tilt_k) > 1:
            kde    = gaussian_kde(tilt_k, bw_method=0.3)
            kdeval = kde(x_range)

            ax.plot(x_range, kdeval,
                    color=ACTIVE_COLORS[i], linewidth=2.0)
            ax.fill_between(x_range, kdeval,
                            color=ACTIVE_COLORS[i], alpha=0.2)

    ax.set_xlabel("Mean tilt angle (°)", fontsize=15)
    ax.set_ylabel("Density", fontsize=15)
    ax.set_title(title, fontsize=16)
    ax.tick_params(axis='both', labelsize=13)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(bottom=0)

# Shared legend
legend_elements = [
    Patch(facecolor=ACTIVE_COLORS[i], alpha=0.6,
          label=f"State {ACTIVE_NAMES[i]}")
    for i in range(N_ACTIVE)
]
fig.legend(handles=legend_elements, loc="lower center",
           ncol=N_ACTIVE, fontsize=13, frameon=True,
           bbox_to_anchor=(0.5, -0.06))

plt.suptitle("Tilt Angle Distributions per State\n"
             "CPSF6-bound vs NUP153 only (rep 0 & rep 1)",
             fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("vampnet_tilt_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved -> vampnet_tilt_distributions.png")

In [ ]:
# ============================================================================
# Curvature (1/R) KDE per state — 2 panels
# Left: CPSF6-bound | Right: NUP153 only
# KDE curves per panel (one per active state), colored by state
# Rep 0 and rep 1 only
# VAMPNet states — active states only, relabeled A-D
# ============================================================================
from scipy.stats import gaussian_kde
from matplotlib.patches import Patch
from scipy.optimize import least_squares

# ── Sphere fitting functions ──────────────────────────────────────────────────
def _sphere_residuals(params, coords):
    cx, cy, cz, R = params
    dists = np.sqrt((coords[:, 0] - cx)**2 +
                    (coords[:, 1] - cy)**2 +
                    (coords[:, 2] - cz)**2)
    return dists - R

def fit_sphere(coords):
    centroid = coords.mean(axis=0)
    R0       = np.linalg.norm(coords - centroid, axis=1).mean()
    x0       = np.array([centroid[0], centroid[1], centroid[2], R0])
    result   = least_squares(_sphere_residuals, x0, args=(coords,), method='lm')
    return abs(result.x[3])

def compute_curvature_traj(traj_path):
    topology, trajectory = traj_path
    print(f"  Computing curvature: {trajectory}")
    u      = mda.Universe(topology, trajectory)
    mobile = u.select_atoms(ALIGN_SEL)
    curvatures = []
    for ts in u.trajectory:
        mob_com      = mobile.positions.mean(axis=0)
        mob_centered = mobile.positions - mob_com
        R_mat        = kabsch_rotation(mob_centered, ref_centered)
        aligned      = mob_centered @ R_mat.T
        R            = fit_sphere(aligned)
        curvatures.append(1.0 / R)
    return np.array(curvatures)

# ── Compute curvature — rep 0 and rep 1 only ─────────────────────────────────
print(f"\n{'='*60}")
print("Computing curvature (1/R) — rep 0 and rep 1 only")
print(f"{'='*60}")

curv_cpsf6_list = [compute_curvature_traj(p) for p in traj_paths[:2]]
curv_bare_list  = [compute_curvature_traj(p) for p in traj_paths[3:5]]

curv_cpsf6_all = np.concatenate(curv_cpsf6_list, axis=0)
curv_bare_all  = np.concatenate(curv_bare_list,  axis=0)

# Use VAMPNet active state dtrajs — rep 0 and rep 1 only
dtrajs_cpsf6_sub = all_dtrajs_active[:3][:2]
dtrajs_bare_sub  = all_dtrajs_active[3:][:2]

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x_min   = min(curv_cpsf6_all.min(), curv_bare_all.min()) - 1e-4
x_max   = max(curv_cpsf6_all.max(), curv_bare_all.max()) + 1e-4
x_range = np.linspace(x_min, x_max, 500)

panel_configs = [
    (curv_cpsf6_all, dtrajs_cpsf6_sub, "CPSF6-bound"),
    (curv_bare_all,  dtrajs_bare_sub,  "NUP153 only"),
]

for ax, (curv_all, dtrajs_sub, title) in zip(axes, panel_configs):
    for i in range(N_ACTIVE):
        dtraj_concat = np.concatenate([d for d in dtrajs_sub], axis=0)
        mask   = dtraj_concat == i
        curv_k = curv_all[mask]

        if len(curv_k) > 1:
            kde    = gaussian_kde(curv_k, bw_method=0.3)
            kdeval = kde(x_range)

            ax.plot(x_range, kdeval,
                    color=ACTIVE_COLORS[i], linewidth=2.0)
            ax.fill_between(x_range, kdeval,
                            color=ACTIVE_COLORS[i], alpha=0.2)

    ax.set_xlabel("Curvature 1/R (Å⁻¹)", fontsize=14)
    ax.set_ylabel("Density", fontsize=14)
    ax.set_title(title, fontsize=14)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(bottom=0)

# Shared legend
legend_elements = [
    Patch(facecolor=ACTIVE_COLORS[i], alpha=0.6,
          label=f"State {ACTIVE_NAMES[i]}")
    for i in range(N_ACTIVE)
]
fig.legend(handles=legend_elements, loc="lower center",
           ncol=N_ACTIVE, fontsize=12, frameon=True,
           bbox_to_anchor=(0.5, -0.04))

plt.suptitle("Curvature (1/R) Distributions per State\n"
             "CPSF6-bound vs NUP153 only (rep 0 & rep 1)",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("vampnet_curvature_kde_per_state.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)